In [ ]:
# >>> pRNA figure routing (added by the pRNA update; keep as the first code cell)
# Every figure is shown inline. Every file written with savefig (png, pdf, svg) is redirected to
# figures_pdf/<notebook name>/ next to this notebook; sub-folders of the original path are preserved.
import os as _fig_os
from pathlib import Path as _FigPath
import matplotlib as _fig_mpl
import matplotlib.pyplot as _fig_plt
from matplotlib.figure import Figure as _FigFigure

NOTEBOOK_NAME = "03. cluster_anlaysis"
FIG_DIR = (_FigPath.cwd() / "figures_pdf" / NOTEBOOK_NAME).resolve()
FIG_DIR.mkdir(parents=True, exist_ok=True)


def _route_figure_path(fname):
    p = _FigPath(_fig_os.fspath(fname))
    full = p if p.is_absolute() else (_FigPath.cwd() / p)
    try:
        full.resolve().relative_to(FIG_DIR)
        return fname                                  # already inside this notebook's figure folder
    except ValueError:
        pass
    try:
        rel = _FigPath(_fig_os.path.relpath(full.resolve(), _FigPath.cwd()))
    except ValueError:                                # e.g. another drive on Windows
        rel = _FigPath(p.name)
    parts = [x for x in rel.parts if x not in ("..", ".", "figures_pdf", NOTEBOOK_NAME)]
    target = FIG_DIR.joinpath(*parts) if parts else FIG_DIR / p.name
    target.parent.mkdir(parents=True, exist_ok=True)
    return target


if not getattr(_FigFigure.savefig, "_prna_routed", False):
    _fig_orig_savefig = _FigFigure.savefig

    def _fig_routed_savefig(self, fname, *args, **kwargs):
        if isinstance(fname, (str, bytes, _FigPath)) or hasattr(fname, "__fspath__"):
            fname = _route_figure_path(fname)
        return _fig_orig_savefig(self, fname, *args, **kwargs)

    _fig_routed_savefig._prna_routed = True
    _FigFigure.savefig = _fig_routed_savefig

    _fig_orig_close = _fig_plt.close

    def _fig_close_and_show(fig=None):
        """Display figures that are closed before Jupyter could render them, then close."""
        import inspect as _fig_inspect
        _caller = _fig_inspect.currentframe().f_back
        _caller_mod = (_caller.f_globals.get("__name__", "") if _caller is not None else "")
        if _caller_mod.startswith(("matplotlib", "ipykernel")):   # backend housekeeping, not user code
            return _fig_orig_close(fig)
        if "inline" in str(_fig_mpl.get_backend()).lower():
            from IPython.display import display as _fig_display
            open_nums = set(_fig_plt.get_fignums())
            if fig is None:
                figs = [_fig_plt.gcf()] if open_nums else []
            elif isinstance(fig, str) and fig == "all":
                figs = [_fig_plt.figure(n) for n in sorted(open_nums)]
            elif isinstance(fig, _FigFigure):
                figs = [fig] if getattr(fig, "number", None) in open_nums else []
            elif isinstance(fig, int):
                figs = [_fig_plt.figure(fig)] if fig in open_nums else []
            else:
                figs = []
            for _f in figs:
                if not getattr(_f, "_prna_displayed", False):
                    _f._prna_displayed = True
                    _fig_display(_f)
        return _fig_orig_close(fig)

    _fig_plt.close = _fig_close_and_show

    _fig_orig_show = _fig_plt.show

    def _fig_show_and_mark(*args, **kwargs):
        for _n in _fig_plt.get_fignums():                    # shown now -> never re-displayed on close
            _fig_plt.figure(_n)._prna_displayed = True
        return _fig_orig_show(*args, **kwargs)

    _fig_plt.show = _fig_show_and_mark

print("Figures are saved to", FIG_DIR)

In [ ]:
#Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import sys
import os
import seaborn as sns
from sklearn.decomposition import PCA
from joblib import dump, load
from collections import defaultdict
from matplotlib.gridspec import GridSpec
from pathlib import Path


# Add the 'src' directory to the system path
src_dir = os.path.abspath(os.path.join(os.getcwd(), "src"))
sys.path.insert(0, src_dir)

#import the module
import corr_utils
from scipy.stats import skew, kurtosis, ks_2samp
from sklearn.metrics import silhouette_samples, silhouette_score



## Datasets

In [ ]:
# Load the full pearson correlation matrix P-G
pearson = load('data/full_pearson100.joblib')
pearson


In [ ]:

transcriptomics_filtered = load('data/transcriptomics_filtered.joblib')
transcriptomics_filtered

In [ ]:
#Remove genes filtered out in the filtering procedure
pearson = pearson[transcriptomics_filtered.columns]
pearson

## Cluster Overlap

In [ ]:

# --- Configuration ---
PG_DIR = 'results/clusters_pg'
PP_DIR = 'results/clusters_pp'
PE_DIR = 'results/clusters_pe'
RESULTS_DIR = Path('results/overlap_analysis')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Read cluster files once ---
pg_clusters = corr_utils.read_cluster_files(PG_DIR)
pp_clusters = corr_utils.read_cluster_files(PP_DIR)
pe_clusters = corr_utils.read_cluster_files(PE_DIR)

# --- Common k values ---
common_k_values = sorted(set(pg_clusters.keys()) & set(pp_clusters.keys()) & set(pe_clusters.keys()))
if not common_k_values:
    print("No common k values found across all three clustering methods!")
else:
    print(f"Common k values found: {common_k_values}")

    # Container for all results so we don't recompute
    results = {}

    # --- Compute pairwise overlaps + consensus once per k and save outputs ---
    for k in common_k_values:
        print(f"\nAnalyzing clusters for k={k}...")

        pg_df = pg_clusters[k]
        pp_df = pp_clusters[k]
        pe_df = pe_clusters[k]

        # Pairwise overlaps (count, percentage, contingency)
        pg_pp_count, pg_pp_pct, pg_pp_cont = corr_utils.compute_overlap(
            pg_df, pp_df,
            df1_name=f"PG_k{k}",
            df2_name=f"PP_k{k}",
            output_file=str(RESULTS_DIR / f"PG_PP_k{k}_overlap.csv")
        )
        pp_pe_count, pp_pe_pct, pp_pe_cont = corr_utils.compute_overlap(
            pp_df, pe_df,
            df1_name=f"PP_k{k}",
            df2_name=f"PE_k{k}",
            output_file=str(RESULTS_DIR / f"PP_PE_k{k}_overlap.csv")
        )
        pg_pe_count, pg_pe_pct, pg_pe_cont = corr_utils.compute_overlap(
            pg_df, pe_df,
            df1_name=f"PG_k{k}",
            df2_name=f"PE_k{k}",
            output_file=str(RESULTS_DIR / f"PG_PE_k{k}_overlap.csv")
        )

        # Save contingency tables
        pg_pp_cont.to_csv(RESULTS_DIR / f"PG_PP_k{k}_contingency.csv")
        pp_pe_cont.to_csv(RESULTS_DIR / f"PP_PE_k{k}_contingency.csv")
        pg_pe_cont.to_csv(RESULTS_DIR / f"PG_PE_k{k}_contingency.csv")

        # Consensus across all three
        consensus = corr_utils.compute_consensus(pg_df, pp_df, pe_df)

        # Store once; re-use for summary & plots
        results[k] = {
            'pg_pp_count': pg_pp_count, 'pg_pp_pct': pg_pp_pct, 'cont_pg_pp': pg_pp_cont,
            'pp_pe_count': pp_pe_count, 'pp_pe_pct': pp_pe_pct, 'cont_pp_pe': pp_pe_cont,
            'pg_pe_count': pg_pe_count, 'pg_pe_pct': pg_pe_pct, 'cont_pg_pe': pg_pe_cont,
            'consensus': consensus,
            # Keep cluster dfs for plotting distributions
            'pg_df': pg_df, 'pp_df': pp_df, 'pe_df': pe_df
        }

        # Console summary for this k
        print(f"Overlap (k={k}): "
              f"PG–PP {pg_pp_count} ({pg_pp_pct:.2f}%), "
              f"PP–PE {pp_pe_count} ({pp_pe_pct:.2f}%), "
              f"PG–PE {pg_pe_count} ({pg_pe_pct:.2f}%).")
        print(f"Full consensus count: {consensus.get('full_consensus_count', 'NA')} "
              f"({consensus.get('all_consensus', 0):.2f}% of total).")

    # --- Summary CSV (counts + percentages + consensus) ---
    summary_rows = []
    for k in common_k_values:
        r = results[k]
        c = r['consensus']
        summary_rows.append({
            'k': k,
            'PG_PP_overlap_count': r['pg_pp_count'],
            'PG_PP_overlap_pct': r['pg_pp_pct'],
            'PP_PE_overlap_count': r['pp_pe_count'],
            'PP_PE_overlap_pct': r['pp_pe_pct'],
            'PG_PE_overlap_count': r['pg_pe_count'],
            'PG_PE_overlap_pct': r['pg_pe_pct'],
            'Full_consensus_count': c.get('full_consensus_count', np.nan),
            'Full_consensus_pct': c.get('all_consensus', np.nan),
            'Total_proteins': c.get('total_proteins', np.nan),
            # (optional) keep the pairwise consensus pcts if your function returns them
            'Pairwise_pg_pp_consensus_pct': c.get('pg_pp_overlap', np.nan),
            'Pairwise_pp_pe_consensus_pct': c.get('pp_pe_overlap', np.nan),
            'Pairwise_pg_pe_consensus_pct': c.get('pg_pe_overlap', np.nan),
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_path = RESULTS_DIR / "overlap_summary.csv"
    summary_df.to_csv(summary_path, index=False)
    print(f"\nOverlap summary saved to {summary_path}")

    # --- Visualization per k: distributions + pairwise heatmaps (2x2 grid) ---
    for k in common_k_values:
        r = results[k]
        pg_df, pp_df, pe_df = r['pg_df'], r['pp_df'], r['pe_df']

        fig = plt.figure(figsize=(20, 15))
        gs = GridSpec(2, 2, figure=fig)

        # Grid 1: Cluster distributions (normalized to %)
        ax1 = fig.add_subplot(gs[0, 0])

        def cluster_pct(df, k_val):
            counts = (df['Cluster']
                      .value_counts()
                      .sort_index()
                      .reindex(range(1, int(k_val) + 1), fill_value=0))
            total = counts.sum()
            return (counts / total * 100) if total else counts.astype(float)

        pg_pct = cluster_pct(pg_df, k)
        pp_pct = cluster_pct(pp_df, k)
        pe_pct = cluster_pct(pe_df, k)

        bar_width = 0.25
        idx = np.arange(1, int(k) + 1)

        ax1.bar(idx - bar_width, pg_pct.values, width=bar_width, label='PG')
        ax1.bar(idx,               pp_pct.values, width=bar_width, label='PP')
        ax1.bar(idx + bar_width,   pe_pct.values, width=bar_width, label='PE')

        ax1.set_xlabel('Cluster Number')
        ax1.set_ylabel('Percentage of Proteins')
        ax1.set_title(f'Cluster Distributions for k={k}')
        ax1.set_xticks(idx)
        ax1.legend()

        # Grid 2: PG vs PP heatmap
        ax2 = fig.add_subplot(gs[0, 1])
        sns.heatmap(r['cont_pg_pp'].astype(int), annot=True, cmap='viridis', fmt='d', ax=ax2)
        ax2.set_title(f'PG vs PP (Overlap: {r["pg_pp_count"]} proteins, {r["pg_pp_pct"]:.2f}%)')
        ax2.set_xlabel('PP Clusters')
        ax2.set_ylabel('PG Clusters')

        # Grid 3: PP vs PE heatmap
        ax3 = fig.add_subplot(gs[1, 0])
        sns.heatmap(r['cont_pp_pe'].astype(int), annot=True, cmap='viridis', fmt='d', ax=ax3)
        ax3.set_title(f'PP vs PE (Overlap: {r["pp_pe_count"]} proteins, {r["pp_pe_pct"]:.2f}%)')
        ax3.set_xlabel('PE Clusters')
        ax3.set_ylabel('PP Clusters')

        # Grid 4: PG vs PE heatmap
        ax4 = fig.add_subplot(gs[1, 1])
        sns.heatmap(r['cont_pg_pe'].astype(int), annot=True, cmap='viridis', fmt='d', ax=ax4)
        ax4.set_title(f'PG vs PE (Overlap: {r["pg_pe_count"]} proteins, {r["pg_pe_pct"]:.2f}%)')
        ax4.set_xlabel('PE Clusters')
        ax4.set_ylabel('PG Clusters')

        plt.tight_layout()
        out_grid = RESULTS_DIR / f'cluster_comparison_k{k}.png'
        plt.savefig(out_grid, dpi=300)
        plt.close()
        print(f"Saved {out_grid}")

        # --- Consensus bar chart ---
        c = r['consensus']
        labels = ['PG–PP', 'PP–PE', 'PG–PE', 'All Three']
        vals = [
            c.get('pg_pp_overlap', np.nan),
            c.get('pp_pe_overlap', np.nan),
            c.get('pg_pe_overlap', np.nan),
            c.get('all_consensus', np.nan)
        ]

        plt.figure(figsize=(10, 8))
        plt.bar(labels, vals)
        plt.ylabel('Percentage of Proteins')
        plt.title(f'Clustering Consensus (k={k})')
        plt.ylim(0, 100)

        for i, v in enumerate(vals):
            if pd.notna(v):
                plt.text(i, min(v + 1, 99.5), f'{v:.2f}%', ha='center', va='bottom')

        total_proteins = c.get('total_proteins', 'NA')
        full_cons_count = c.get('full_consensus_count', 'NA')
        all_cons_pct = c.get('all_consensus', np.nan)
        ann_text = (f'Total proteins: {total_proteins}\n'
                    f'Full consensus count: {full_cons_count}'
                    + (f' ({all_cons_pct:.2f}%)' if pd.notna(all_cons_pct) else ''))

        plt.annotate(
            ann_text, xy=(0.5, 0.9), xycoords='axes fraction',
            ha='center',
            bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8)
        )

        plt.tight_layout()
        out_cons = RESULTS_DIR / f'consensus_visualization_k{k}.png'
        plt.savefig(out_cons, dpi=300)
        plt.close()
        print(f"Saved {out_cons}")

    print("\nDone.")


In [ ]:
# --- Visualize results (updated to match the single-cell 'results' structure) ---
for k in common_k_values:
    r = results[k]
    pg_df, pp_df, pe_df = r['pg_df'], r['pp_df'], r['pe_df']

    # Create figure with subplots
    fig = plt.figure(figsize=(20, 15))
    gs = GridSpec(2, 2, figure=fig)

    # -------- Grid 1: Individual cluster distributions --------
    ax1 = fig.add_subplot(gs[0, 0])

    def cluster_pct(df, k_val):
        counts = (df['Cluster']
                  .value_counts()
                  .sort_index()
                  .reindex(range(1, int(k_val) + 1), fill_value=0))
        total = counts.sum()
        return (counts / total * 100) if total else counts.astype(float)

    pg_cluster_pct = cluster_pct(pg_df, k)
    pp_cluster_pct = cluster_pct(pp_df, k)
    pe_cluster_pct = cluster_pct(pe_df, k)

    bar_width = 0.25
    indices = np.arange(1, int(k) + 1)

    ax1.bar(indices - bar_width, pg_cluster_pct.values, bar_width, label='PG')
    ax1.bar(indices,               pp_cluster_pct.values, bar_width, label='PP')
    ax1.bar(indices + bar_width,   pe_cluster_pct.values, bar_width, label='PE')

    ax1.set_xlabel('Cluster Number')
    ax1.set_ylabel('Percentage of Proteins')
    ax1.set_title(f'Cluster Distributions for k={k}')
    ax1.set_xticks(indices)
    ax1.legend()

    # -------- Grid 2: PG vs PP pairwise comparison --------
    ax2 = fig.add_subplot(gs[0, 1])
    sns.heatmap(r['cont_pg_pp'].astype(int), annot=True, cmap='viridis', fmt='d', ax=ax2)
    ax2.set_title(f'PG vs PP Clusters (Overlap: {r["pg_pp_count"]} proteins, {r["pg_pp_pct"]:.2f}%)')
    ax2.set_xlabel('PP Clusters')
    ax2.set_ylabel('PG Clusters')

    # -------- Grid 3: PP vs PE pairwise comparison --------
    ax3 = fig.add_subplot(gs[1, 0])
    sns.heatmap(r['cont_pp_pe'].astype(int), annot=True, cmap='viridis', fmt='d', ax=ax3)
    ax3.set_title(f'PP vs PE Clusters (Overlap: {r["pp_pe_count"]} proteins, {r["pp_pe_pct"]:.2f}%)')
    ax3.set_xlabel('PE Clusters')
    ax3.set_ylabel('PP Clusters')

    # -------- Grid 4: PG vs PE pairwise comparison --------
    ax4 = fig.add_subplot(gs[1, 1])
    sns.heatmap(r['cont_pg_pe'].astype(int), annot=True, cmap='viridis', fmt='d', ax=ax4)
    ax4.set_title(f'PG vs PE Clusters (Overlap: {r["pg_pe_count"]} proteins, {r["pg_pe_pct"]:.2f}%)')
    ax4.set_xlabel('PE Clusters')
    ax4.set_ylabel('PG Clusters')

    plt.tight_layout()
    grid_path = RESULTS_DIR / f'cluster_comparison_k{k}.png'
    plt.savefig(grid_path, dpi=300)
    plt.close()
    print(f"Saved {grid_path}")

    # -------- Additional figure: consensus visualization --------
    cons = r['consensus']
    labels = ['PG-PP', 'PP-PE', 'PG-PE', 'All Three']
    consensus_data = [
        cons.get('pg_pp_overlap', np.nan),
        cons.get('pp_pe_overlap', np.nan),
        cons.get('pg_pe_overlap', np.nan),
        cons.get('all_consensus', np.nan)
    ]

    fig_consensus = plt.figure(figsize=(10, 8))
    plt.bar(labels, consensus_data)
    plt.ylabel('Percentage of Proteins')
    plt.title(f'Clustering Consensus (k={k})')
    plt.ylim(0, 100)

    for i, v in enumerate(consensus_data):
        if pd.notna(v):
            plt.text(i, min(v + 1, 99.5), f'{v:.2f}%', ha='center', va='bottom')

    total_proteins = cons.get('total_proteins', 'NA')
    full_consensus_count = cons.get('full_consensus_count', 'NA')
    all_consensus_pct = cons.get('all_consensus', np.nan)
    ann = (f'Total proteins: {total_proteins}\n'
           f'Full consensus count: {full_consensus_count}'
           + (f' ({all_consensus_pct:.2f}%)' if pd.notna(all_consensus_pct) else ''))

    plt.annotate(
        ann,
        xy=(0.5, 0.9), xycoords='axes fraction', ha='center',
        bbox=dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.8)
    )

    plt.tight_layout()
    cons_path = RESULTS_DIR / f'consensus_visualization_k{k}.png'
    plt.savefig(cons_path, dpi=300)
    plt.close(fig_consensus)
    print(f"Saved {cons_path}")


In [ ]:
# --- Print summary of results (updated to new 'results' structure) ---
print("\nSummary of Cluster Overlaps:")
print("=" * 50)
print(f"{'k':<5}{'PG-PP %':>10}{'PP-PE %':>10}{'PG-PE %':>10}{'All Three %':>15}")
print("-" * 50)

for k in sorted(results.keys()):
    pg_pp_pct = results[k]['pg_pp_pct']
    pp_pe_pct = results[k]['pp_pe_pct']
    pg_pe_pct = results[k]['pg_pe_pct']
    all_three_pct = results[k]['consensus'].get('all_consensus', float('nan'))
    print(f"{k:<5}{pg_pp_pct:>10.2f}{pp_pe_pct:>10.2f}{pg_pe_pct:>10.2f}{all_three_pct:>15.2f}")

print("\nDetailed Consensus Statistics:")
print("=" * 70)
for k in sorted(results.keys()):
    cons = results[k]['consensus']
    total = cons.get('total_proteins', 'NA')
    full_cnt = cons.get('full_consensus_count', 'NA')
    all_pct = cons.get('all_consensus', float('nan'))
    pg_pp = cons.get('pg_pp_overlap', float('nan'))
    pp_pe = cons.get('pp_pe_overlap', float('nan'))
    pg_pe = cons.get('pg_pe_overlap', float('nan'))

    print(f"\nk = {k}:")
    print(f"  Total proteins analyzed: {total}")
    try:
        print(f"  Full consensus count: {int(full_cnt)} proteins ({float(all_pct):.2f}%)")
    except Exception:
        print(f"  Full consensus count: {full_cnt} proteins")

    try:
        print(f"  PG-PP overlap: {float(pg_pp):.2f}%")
    except Exception:
        print(f"  PG-PP overlap: {pg_pp}")

    try:
        print(f"  PP-PE overlap: {float(pp_pe):.2f}%")
    except Exception:
        print(f"  PP-PE overlap: {pp_pe}")

    try:
        print(f"  PG-PE overlap: {float(pg_pe):.2f}%")
    except Exception:
        print(f"  PG-PE overlap: {pg_pe}")


#### Save results

In [ ]:
# --- Build & save comprehensive results tables (no plotting) ---
# Uses the unified 'results' structure with pg_pp_pct/pp_pe_pct/pg_pe_pct and 'consensus'
rows = []
for k in sorted(results.keys()):
    cons = results[k]['consensus']
    pg_pp_pct = results[k]['pg_pp_pct']
    pp_pe_pct = results[k]['pp_pe_pct']
    pg_pe_pct = results[k]['pg_pe_pct']
    avg_pairwise_pct = (pg_pp_pct + pp_pe_pct + pg_pe_pct) / 3.0

    rows.append({
        'k': k,
        'Total_Proteins': cons.get('total_proteins'),
        'PG_PP_Overlap_pct': pg_pp_pct,
        'PP_PE_Overlap_pct': pp_pe_pct,
        'PG_PE_Overlap_pct': pg_pe_pct,
        'Avg_Pairwise_Overlap_pct': avg_pairwise_pct,
        'Full_Consensus_pct': cons.get('all_consensus'),
        'Full_Consensus_Count': cons.get('full_consensus_count'),
        'PG_PP_Overlap_count': results[k]['pg_pp_count'],
        'PP_PE_Overlap_count': results[k]['pp_pe_count'],
        'PG_PE_Overlap_count': results[k]['pg_pe_count'],
    })

results_df = pd.DataFrame(rows, columns=[
    'k',
    'Total_Proteins',
    'PG_PP_Overlap_pct', 'PP_PE_Overlap_pct', 'PG_PE_Overlap_pct',
    'Avg_Pairwise_Overlap_pct',
    'Full_Consensus_pct', 'Full_Consensus_Count',
    'PG_PP_Overlap_count', 'PP_PE_Overlap_count', 'PG_PE_Overlap_count',
])

out_csv = RESULTS_DIR / "cluster_overlap_results.csv"
results_df.to_csv(out_csv, index=False)
print(f"\nCluster overlap results saved to '{out_csv}'")

print("\nSummary of Cluster Overlaps:")
print("=" * 80)
with pd.option_context('display.float_format', lambda x: f"{x:.2f}"):
    print(results_df.to_string(index=False))


# Protein - Gene Threshold setting 2-3 

## 2 clusters


In [ ]:
clusters = pd.read_csv('results/clusters_pg/protein_clusters_k2.csv')
clusters3 = pd.read_csv('results/clusters_pg/protein_clusters_k3.csv')

In [ ]:
prot_c1 = clusters.loc[clusters.Cluster == 1, "Protein"]
prot_c2 = clusters.loc[clusters.Cluster == 2, "Protein"]

df_cluster1 = pearson.loc[pearson.index.intersection(prot_c1)]
df_cluster2 = pearson.loc[pearson.index.intersection(prot_c2)]

In [ ]:
df_cluster1.describe()


In [ ]:
df_cluster2.describe()

In [ ]:

# compute the describe() tables
desc1 = df_cluster1.describe()
desc2 = df_cluster2.describe()

# pull out the "max" row and take its mean
avg_of_max1 = desc1.loc["max"].mean()
avg_of_max2 = desc2.loc["max"].mean()

# print the results
print(f"Average of per‑column max correlations (Cluster 1): {avg_of_max1:.4f}")
print(f"Average of per‑column max correlations (Cluster 2): {avg_of_max2:.4f}")


Distributional analysis

In [ ]:
# 2. Flatten and clean correlation values for distributional analysis
vals1 = df_cluster1.values.flatten()
vals2 = df_cluster2.values.flatten()
# Drop NaNs
vals1 = vals1[~np.isnan(vals1)]
vals2 = vals2[~np.isnan(vals2)]

# %% [code]
# 3. Summary statistics & percentiles
for name, vals in [('Cluster 1', vals1), ('Cluster 2', vals2)]:
    print(f"{name} Summary:")
    print(f"  Mean       = {np.mean(vals):.4f}")
    print(f"  Std Dev    = {np.std(vals, ddof=1):.4f}")
    print(f"  Median     = {np.median(vals):.4f}")
    print(f"  Skewness   = {skew(vals):.4f}")
    print(f"  Kurtosis   = {kurtosis(vals):.4f}")
    for p in [75, 90, 95]:
        print(f"  {p}th pct  = {np.percentile(vals, p):.4f}")
    print()

Correlation Thresholds

In [ ]:
# 4. Compute cluster-specific correlation thresholds (e.g., top 10% strongest absolute correlations)
thr_pct = 90  # adjust percentile as needed
thr1 = np.percentile(np.abs(vals1), thr_pct)
thr2 = np.percentile(np.abs(vals2), thr_pct)
print(f"Threshold for top {thr_pct}% |r| in Cluster 1: {thr1:.4f}")
print(f"Threshold for top {thr_pct}% |r| in Cluster 2: {thr2:.4f}")

### Find critical thresholds

Exceed threshold in all proteins

In [ ]:
# --- Config ---
percentiles = np.linspace(50, 100, 50)

# Store results
thresholds = {}
gene_counts = {}

# First populate the thresholds and gene_counts dictionaries
for i, (name, df_cluster) in enumerate({"Cluster 1": df_cluster1, "Cluster 2": df_cluster2}.items()):
    vals = df_cluster.values.flatten()
    vals = vals[~np.isnan(vals)]
    
    thr_values = np.percentile(np.abs(vals), percentiles)
    thresholds[name] = thr_values
    
    counts = []
    for thr in thr_values:
        # Calculate average number of genes above threshold per protein (matching graph calculation)
        mask = (np.abs(df_cluster.values) >= thr)
        avg_genes_per_protein = mask.sum(axis=1).mean()
        counts.append(avg_genes_per_protein)
    gene_counts[name] = counts

# Now calculate critical points for each cluster
for name in thresholds.keys():
    # Calculate rate of change (derivative)
    gene_counts_array = np.array(gene_counts[name])
    derivatives = np.diff(gene_counts_array) / np.diff(percentiles)
    
    # Find the index of maximum rate of change (most negative value)
    critical_idx = np.argmin(derivatives)
    
    # Get corresponding percentile and gene counts
    critical_percentile = percentiles[critical_idx]
    critical_percentile_next = percentiles[critical_idx + 1]
    genes_at_critical = gene_counts[name][critical_idx]
    genes_at_next = gene_counts[name][critical_idx + 1]
    
    print(f"\n{name} critical point:")
    print(f"  Percentile range: {critical_percentile:.2f} - {critical_percentile_next:.2f}")
    print(f"  Avg genes per protein: {genes_at_critical:.2f} → {genes_at_next:.2f}")
    print(f"  Drop magnitude: {genes_at_critical - genes_at_next:.2f}")
    
    # Find threshold value at the critical point
    threshold_value = thresholds[name][critical_idx]
    print(f"  Correlation threshold value: {threshold_value:.4f}")
    
    # Optional: plot the derivatives to visualize the critical point
    plt.figure(figsize=(8, 4))
    plt.plot(percentiles[:-1], derivatives)
    plt.axvline(x=critical_percentile, color='r', linestyle='--')
    plt.title(f"{name}: Rate of Change in Gene Count")
    plt.xlabel("Percentile Threshold")
    plt.ylabel("Rate of Change")
    plt.grid(True)
    plt.show()

# Bonus: Plot the gene counts vs percentiles with critical points marked
plt.figure(figsize=(12, 6))
for name in thresholds.keys():
    plt.plot(percentiles, gene_counts[name], label=name)
    
    # Get critical point index and mark it on the plot
    critical_idx = np.argmin(np.diff(gene_counts[name]) / np.diff(percentiles))
    plt.scatter(percentiles[critical_idx], gene_counts[name][critical_idx], 
                marker='o', s=100, edgecolors='black', zorder=5)

plt.xlabel("Percentile Threshold")
plt.ylabel("Average Number of Genes Selected per Protein")
plt.title("Genes Remaining vs Correlation Percentile Threshold")
plt.legend()
plt.grid(True)
plt.show()

## 3 clusters

In [ ]:
prot_c1 = clusters3.loc[clusters3.Cluster == 1, "Protein"]
prot_c2 = clusters3.loc[clusters3.Cluster == 2, "Protein"]
prot_c3 = clusters3.loc[clusters3.Cluster == 3, "Protein"] 

df_cluster1 = pearson.loc[pearson.index.intersection(prot_c1)]
df_cluster2 = pearson.loc[pearson.index.intersection(prot_c2)]
df_cluster3 = pearson.loc[pearson.index.intersection(prot_c3)]
# now df_cluster1 and df_cluster2 each contain only the rows
# for proteins in cluster 1 or cluster 2, respectively.

In [ ]:
df_cluster1.describe()


In [ ]:
df_cluster2.describe()

In [ ]:
df_cluster3.describe()

In [ ]:

# compute the describe() tables
desc1 = df_cluster1.describe()
desc2 = df_cluster2.describe()
desc3 = df_cluster3.describe()

# pull out the "max" row and take its mean
avg_of_max1 = desc1.loc["max"].mean()
avg_of_max2 = desc2.loc["max"].mean()
avg_of_max3 = desc3.loc["max"].mean()

# print the results
print(f"Average of per‑column max correlations (Cluster 1): {avg_of_max1:.4f}")
print(f"Average of per‑column max correlations (Cluster 2): {avg_of_max2:.4f}")
print(f"Average of per‑column max correlations (Cluster 3): {avg_of_max3:.4f}")


We choose this number of genes per cluster by sweeping percentile cutoffs on |r| within that cluster, producing a curve of the average genes retained per protein as the threshold tightens.
We then locate the elbow—the percentile with the steepest drop—where increasing strictness begins to discard many genes at once, signaling diminishing returns.
Selecting the |r| at this elbow preserves the strongest, most reliable correlations while avoiding over-pruning that would remove useful signal.
We convert that elbow threshold into the expected genes retained per protein and use it as the cluster’s gene count, giving a data-adaptive and comparable choice across clusters.

We start by defining two knobs: a percentile cutoff thr_pct (default 90th) that becomes a cluster-specific |r| threshold, and k_top_genes (how many “driver” genes we report). We package the three cluster matrices (df_cluster1/2/3), same gene columns, rows as proteins/samples—with entries that are typically Pearson correlations r.

We then flatten each cluster’s entries to study its correlation distribution (mean, median, SD, skewness, kurtosis, and key percentiles like 75/90/95). From these distributions, we set a cluster-specific cutoff as the thr_pct-th percentile of |r|; clusters with broader tails naturally get a higher absolute threshold, so the rule adapts to each cluster’s signal strength.

To choose how strict the cutoff should be, we run two complementary analyses. First, a coarse sweep (percentiles_to_test) prints how many genes are retained at several percentiles (corr_utils.compute_gene_retention). Second, a dense sweep (50→100%) maps percentile → average genes per protein retained (count with |r| ≥ threshold) for each cluster. We take the discrete derivative of this curve and pick the most negative slope, the elbow, where tightening the threshold suddenly removes many genes. We report that critical percentile, its |r| value, and the expected genes per protein retained, and we plot both the derivative (with the elbow) and the “genes-remaining vs percentile” curve (with the elbow marked).

In practice, we set the selection cutoff to the cluster-specific |r| at the elbow. This preserves the strongest associations while avoiding over-pruning. The corresponding average genes per protein gives a practical proxy for how many genes to keep in each cluster. Direction is ignored because we threshold |r|; it’s set that all clusters share the same gene set/order; NaNs are handled via np.nan* operations; and, if the derivative is noisy, a light smoothing on the gene-count curve can be applied so the elbow is more stable.

In [ ]:
# --- Configuration ---
thr_pct = 90  # percentile for threshold selection
k_top_genes = 20  # top driver genes

# --- Prepare data ---
clusters = {
    'Cluster 1': df_cluster1,
    'Cluster 2': df_cluster2,
    'Cluster 3': df_cluster3
}

# Flatten and clean values
vals = {}
for name, df in clusters.items():
    v = df.values.flatten()
    v = v[~np.isnan(v)]
    vals[name] = v

# --- Summary Statistics ---
print("=== Summary Statistics ===")
for name, v in vals.items():
    print(f"\n{name}:")
    print(f"  Mean       = {np.mean(v):.4f}")
    print(f"  Std Dev    = {np.std(v, ddof=1):.4f}")
    print(f"  Median     = {np.median(v):.4f}")
    print(f"  Skewness   = {skew(v):.4f}")
    print(f"  Kurtosis   = {kurtosis(v):.4f}")
    for p in [75, 90, 95]:
        print(f"  {p}th pct  = {np.percentile(v, p):.4f}")

# --- Thresholds ---
print("\n=== Cluster-specific Thresholds (|r| at top {}%) ===".format(thr_pct))
thresholds = {}
for name, v in vals.items():
    thr = np.percentile(np.abs(v), thr_pct)
    thresholds[name] = thr
    print(f"{name}: {thr:.4f}")

# --- Cohesion & Separation Metrics ---
print("\n=== Cohesion, Separation, and Silhouette Scores ===")
df_all = pd.concat(list(clusters.values()))
labels = np.concatenate([[i+1]*df.shape[0] for i, df in enumerate(clusters.values())])
X = df_all.values

# Centroids
centroids = {i+1: np.nanmean(df.values, axis=0) for i, df in enumerate(clusters.values())}
global_centroid = np.nanmean(X, axis=0)

# WSS
wss = {}
for i, (name, df) in enumerate(clusters.items(), start=1):
    diffs = df.values - centroids[i]
    wss[i] = np.nansum((diffs**2).sum(axis=1))
    print(f"WSS {name}: {wss[i]:.2f}")

# BSS
bss = sum(df.shape[0] * np.sum((centroids[i+1] - global_centroid)**2) for i, df in enumerate(clusters.values()))
print(f"Between-Cluster SS (BSS): {bss:.2f}")

# Silhouette
sil_vals = silhouette_samples(X, labels, metric='euclidean')
sil_mean = silhouette_score(X, labels, metric='euclidean')
print(f"Mean Silhouette Score: {sil_mean:.4f}")
for i in range(1, 4):
    cluster_sil = sil_vals[labels == i].mean()
    print(f"Cluster {i} Silhouette: {cluster_sil:.4f}")

# --- Cluster Centroids & Top Driver Genes ---
print("\n=== Top Driver Genes per Cluster ===")
genes = df_cluster1.columns  # assuming same columns across clusters
for i in range(1, 4):
    cent = centroids[i]
    top_genes = genes[np.argsort(np.abs(cent))[-k_top_genes:]][::-1]
    print(f"\nTop {k_top_genes} genes for Cluster {i}:")
    print(top_genes.tolist())

In [ ]:
# Main analysis
# Threshold-based gene retention analysis

# --- Configuration ---
percentiles_to_test = [75, 85, 90, 95, 97]
clusters = {
    'Cluster 1': df_cluster1,
    'Cluster 2': df_cluster2,
    'Cluster 3': df_cluster3,

}
for cluster_name, df_c in clusters.items():
    print(f"\n=== {cluster_name} ===")
    thresholds, retained = corr_utils.compute_gene_retention(df_c, percentiles_to_test)
    for p in percentiles_to_test:
        print(f"Percentile {p}% (Threshold: {thresholds[p]:.4f}): {retained[p]} genes retained")


In [ ]:
# --- Config ---
percentiles = np.linspace(50, 100, 50)

# Store results
thresholds = {}
gene_counts = {}

# First populate the thresholds and gene_counts dictionaries
for i, (name, df_cluster) in enumerate({"Cluster 1": df_cluster1, "Cluster 2": df_cluster2, "Cluster 3": df_cluster3}.items()):
    vals = df_cluster.values.flatten()
    vals = vals[~np.isnan(vals)]
    
    thr_values = np.percentile(np.abs(vals), percentiles)
    thresholds[name] = thr_values
    
    counts = []
    for thr in thr_values:
        # Calculate average number of genes above threshold per protein (matching graph calculation)
        mask = (np.abs(df_cluster.values) >= thr)
        avg_genes_per_protein = mask.sum(axis=1).mean()
        counts.append(avg_genes_per_protein)
    gene_counts[name] = counts

# Now calculate critical points for each cluster
for name in thresholds.keys():
    # Calculate rate of change (derivative)
    gene_counts_array = np.array(gene_counts[name])
    derivatives = np.diff(gene_counts_array) / np.diff(percentiles)
    
    # Find the index of maximum rate of change (most negative value)
    critical_idx = np.argmin(derivatives)
    
    # Get corresponding percentile and gene counts
    critical_percentile = percentiles[critical_idx]
    critical_percentile_next = percentiles[critical_idx + 1]
    genes_at_critical = gene_counts[name][critical_idx]
    genes_at_next = gene_counts[name][critical_idx + 1]
    
    print(f"\n{name} critical point:")
    print(f"  Percentile range: {critical_percentile:.2f} - {critical_percentile_next:.2f}")
    print(f"  Avg genes per protein: {genes_at_critical:.2f} → {genes_at_next:.2f}")
    print(f"  Drop magnitude: {genes_at_critical - genes_at_next:.2f}")
    
    # Find threshold value at the critical point
    threshold_value = thresholds[name][critical_idx]
    print(f"  Correlation threshold value: {threshold_value:.4f}")
    
    # Optional: plot the derivatives to visualize the critical point
    plt.figure(figsize=(8, 4))
    plt.plot(percentiles[:-1], derivatives)
    plt.axvline(x=critical_percentile, color='r', linestyle='--')
    plt.title(f"{name}: Rate of Change in Gene Count")
    plt.xlabel("Percentile Threshold")
    plt.ylabel("Rate of Change")
    plt.grid(True)
    plt.show()

# Bonus: Plot the gene counts vs percentiles with critical points marked
plt.figure(figsize=(12, 6))
for name in thresholds.keys():
    plt.plot(percentiles, gene_counts[name], label=name)
    
    # Get critical point index and mark it on the plot
    critical_idx = np.argmin(np.diff(gene_counts[name]) / np.diff(percentiles))
    plt.scatter(percentiles[critical_idx], gene_counts[name][critical_idx], 
                marker='o', s=100, edgecolors='black', zorder=5)

plt.xlabel("Percentile Threshold")
plt.ylabel("Average Number of Genes Selected per Protein")
plt.title("Genes Remaining vs Correlation Percentile Threshold")
plt.legend()
plt.grid(True)
plt.show()